In [ ]:
import pandas as pd

In [ ]:
bein_src = pd.read_csv(r"S:\24.08.26\30800329_BEINDATANEWRPT.CSV", dtype='str')

In [ ]:
bein = bein_src.copy()
bein['End Date'] = pd.to_datetime(bein['End Date'], dayfirst=True,errors='coerce')
bein['Customer Number'] = pd.to_numeric(bein['Customer Number']) +1014
bein['Decoder'] = pd.to_numeric(bein['Decoder']) +1014
bein['Smart Card'] = pd.to_numeric(bein['Smart Card']) +1014

In [ ]:
bein['Customer Type'].unique()

In [ ]:
bein.columns

In [ ]:
freebox = bein.copy()
freebox = freebox.loc[freebox['Plan'].str.contains('frebx',case=False)]

In [ ]:
freebox = freebox.drop_duplicates(subset=['Customer Number'])

In [ ]:
main_pcks = bein.copy()
main_pcks = main_pcks.loc[(main_pcks['Plan'].str.contains('prem',case=False)) 
                          | (main_pcks['Plan'].str.contains('ulti',case=False)) 
                          | (main_pcks['Plan'].str.contains('toget',case=False)) 
                          | (main_pcks['Plan'].str.contains('kick',case=False))
                          | (main_pcks['Plan'].str.contains('beIN Sports',case=False))]

In [ ]:
main_grouped = main_pcks.groupby(['Customer Number']).agg(max_date = ('End Date','max')).reset_index()
main_grouped

In [ ]:
main_grouped = main_grouped.loc[main_grouped['Customer Number'].isin(freebox['Customer Number'])]

In [ ]:
main_grouped

In [ ]:
recent_prods = pd.merge(left= main_grouped, right= main_pcks, left_on=['Customer Number','max_date'], right_on=['Customer Number','End Date'] , how = 'inner')

recent_prods = recent_prods.drop_duplicates(subset=['Customer Number','Plan'])

recent_prods = recent_prods.sort_values(['Customer Number','Plan'], ascending=[True,True])
recent_prods = recent_prods.drop_duplicates('Customer Number', keep='first')


In [ ]:
recent_prods

In [ ]:
g =  recent_prods.groupby('Customer Number').agg(count = ('Customer Number','count')).reset_index()
g.loc[g['count']>1]

In [ ]:
recent_prods.loc[recent_prods['Customer Number']=='19187066']

In [ ]:
freebox.to_csv('freebox.csv', index=False)
recent_prods.to_csv('recent_prods.csv', index=False)

final = pd.merge(left=recent_prods, right=freebox[['Customer Number','Plan']], on='Customer Number').rename(columns={'Plan_y':'Offer', 'Plan_x':'Plan'})
final['Main pck']=''
final.loc[final['Plan'].str.contains('prem',case=False), 'Main pck'] = 'Premium'
final.loc[final['Plan'].str.contains('ulti',case=False), 'Main pck'] = 'Ultimate'
final.loc[final['Plan'].str.contains('toge',case=False), 'Main pck'] = 'Together'
final.loc[final['Plan'].str.contains('kick',case=False), 'Main pck'] = 'Kickoff'


final['Offer pck']=''
final.loc[final['Offer'].str.contains('prem',case=False), 'Offer pck'] = 'Premium'
final.loc[final['Offer'].str.contains('ulti',case=False), 'Offer pck'] = 'Ultimate'
final.loc[final['Offer'].str.contains('toge',case=False), 'Offer pck'] = 'Together'
final.loc[final['Offer'].str.contains('kick',case=False), 'Offer pck'] = 'Kickoff'


final['Offer type']=''
final.loc[final['Offer'].str.contains('ess',case=False), 'Offer type'] = 'Essential'
final.loc[final['Offer'].str.contains('vip',case=False), 'Offer type'] = 'VIP'
final.loc[final['Offer'].str.contains('adv',case=False), 'Offer type'] = 'Advanced'


final.loc[final['Status'].str.contains('DIS',case=False), 'Status'] = 'Disconnected'


In [ ]:
final.to_csv('freebox final.csv', index=False)

In [ ]:
final['Customer Type'].unique()

In [ ]:
final.columns

In [ ]:
full_data = bein.loc[bein['Customer Number'].isin(freebox['Customer Number'])]
full_data = full_data.sort_values(['Customer Number','Plan'], ascending=[True,True])
full_data = full_data.drop_duplicates(subset=['Customer Number','Plan'], keep='first')

In [ ]:
full_data.to_csv('freebox offer full data.csv', index= False)